# 🎲 Die Suche nach der "besten" Setzstrategie (zwei Würfel)

Dieses Programm berechnet **exakt** die Gewinnwahrscheinlichkeiten  
$$ P_A(S,T),\ P_B(S,T) $$
und gibt sämtliche Setzstrategien aus, die "besser" sind als $S$, für die also gilt 
$$P_B(S,T)>P_A(S,T).$$

Zusätzich wird die Laufzeit und die Anzahl aller vergleichender Strategien ausgegeben.

Da die Anzahl an möglichen Setzstrategien schnell anwächst, können jeweils Begrenzungen für die Anzahl an Chips auf den Feldern festgesetzt werden.

---

### ✅ Voraussetzungen

- Es werden genau **zwei Würfel** mit benutzerdefinierten Wahrscheinlichkeiten verwendet, die sich zu 1 summieren.

---

### ⚙️ Eingaben

- `p`: die Wahrscheinlichkeiten für die einzelnen Felder.  
- `S`: Basis-Setzstrategie
- `min_limits`: Mindestanzahl an Chips auf den einzelnen Feldern 
- `max_limits`: Maximalanzahl an Chips auf den einzelnen Feldern 
   
---

### ▶️ So geht’s weiter

In der **nächsten Code-Zelle** befindet sich das zugehörige Python-Programm.  
Dort müssen lediglich folgende Eingaben angepasst werden:

- `p`: Wahrscheinlichkeiten für die Felder (z. B. `p = (1/2, 1/3, 1/6)`)
- `S`: Basis-Setzstrategie  (z. B. `S = (3, 2, 1)`)
- `min_limits`: Mindestanzahl an Chips auf den einzelnen Feldern (z.B. `min_limits = (0, 0, 0)`)
- `max_limits`: Maximalanzahl an Chips auf den einzelnen Feldern (z.B. `max_limits = (6, 6, 6)`) 


> 📌 Achten Sie darauf, dass `p`, `V`, `min_limits`und `max_limits` gleich lang sind!

>  📌 Vorsicht: Die Rechenzeit vergrößert sich schnell.
---

### ▶️ Ausführen des Programms

- Klicken Sie in die Code-Zelle.
- Drücken Sie `Shift + Enter`, um die Berechnung zu starten.  
  Alternativ können Sie auch auf **▶ Run** oben in der Werkzeugleiste klicken.
- Über **Run → Run all cells** werden alle Zellen auf einmal ausgeführt.


📎 Viel Erfolg beim Experimentieren mit eigenen Strategien!

---

In [1]:
import itertools
from functools import lru_cache
from typing import Tuple, List
import time

# ====== Eingabe: Basisstrategie und Wahrscheinlichkeiten ======

# ====== Würfel trifft =====
S = (3, 2, 1)
p = (1/2, 1/3, 1/6)
# Limits definieren
min_limits = (0, 0, 0)
max_limits = (6, 6, 6)

# ==== Differenz trifft ======
#S = (3, 6, 5, 3, 1, 0)
#p = [3/18, 5/18, 4/18, 3/18, 2/18, 1/18] 
# Limits definieren
# min_limits = (3, 5, 4, 3, 0, 0)
# max_limits = (4, 7, 5, 4, 2, 1)


# ===== Rekursionsfunktionen  =====
def make_two_dice_recursion(p1_list: List[float], p2_list: List[float]):
    @lru_cache(maxsize=None)
    def P_A2(V: Tuple[int,...], W: Tuple[int,...]) -> float:
        V = list(V); W = list(W)
        if sum(V) == 0 and sum(W) > 0:
            return 1.0
        if sum(W) == 0:
            return 0.0

        X = 0.0
        p_stay = 0.0
        m = len(p1_list)
        for i in range(m):
            for j in range(m):
                pij = p1_list[i] * p2_list[j]
                a_rem = V[i] > 0
                b_rem = W[j] > 0
                if not (a_rem or b_rem):
                    p_stay += pij
                else:
                    V2 = V.copy(); W2 = W.copy()
                    if a_rem: V2[i] -= 1
                    if b_rem: W2[j] -= 1
                    s2, t2 = sum(V2), sum(W2)
                    if s2 == 0 and t2 > 0:
                        X += pij
                    elif t2 == 0:
                        pass
                    else:
                        X += pij * P_A2(tuple(V2), tuple(W2))
        return X / (1.0 - p_stay)

    return P_A2

def compute_two_dice(V: Tuple[int,...],
                     W: Tuple[int,...],
                     p1_list: List[float],
                     p2_list: List[float]):
    P_A2 = make_two_dice_recursion(p1_list, p2_list)
    PA = P_A2(tuple(V), tuple(W))
    PB = P_A2(tuple(W), tuple(V))
    PU = 1.0 - PA - PB
    return PA, PB, PU

# ===== Neue Funktion mit Limits =====
def generate_strategies_with_limits(total_chips: int,
                                    num_fields: int,
                                    min_limits: Tuple[int,...],
                                    max_limits: Tuple[int,...]):
    """
    Erzeugt alle Strategien T (Länge num_fields, Summe total_chips),
    wobei für jedes Feld i gilt: min_limits[i] <= T[i] <= max_limits[i].
    """
    for combo in itertools.combinations_with_replacement(range(num_fields), total_chips):
        strat = [0] * num_fields
        for idx in combo:
            strat[idx] += 1
        # Filter nach min/max
        valid = True
        for i in range(num_fields):
            if not (min_limits[i] <= strat[i] <= max_limits[i]):
                valid = False
                break
        if valid:
            yield tuple(strat)

# --- Suche nach besseren Strategien ---
def find_better_strategies(S: Tuple[int,...],
                           p: List[float],
                           min_limits: Tuple[int,...],
                           max_limits: Tuple[int,...]):
    total = sum(S)
    m = len(S)
    better = []
    count = 0
    for T in generate_strategies_with_limits(total, m, min_limits, max_limits):
        count += 1
        PA, PB, _ = compute_two_dice(S, T, p, p)
        if PA < PB:
            better.append((T, PA, PB))
    return better, count

# Suche starten
t2 = time.perf_counter()
better_strats, total_comparisons = find_better_strategies(S, p, min_limits, max_limits)
t3 = time.perf_counter()
print(f"Laufzeit: {t3-t2:.3f}s (zwei Würfel)")
print(f"Basisstrategie S = {S}")
print(f"Insgesamt zu vergleichende Strategien: {total_comparisons}")
print(f"Gefundene bessere Strategien: {len(better_strats)}\n")
for T, PA, PB in sorted(better_strats, key=lambda x: x[1]):
        T_str = str(T)
        print(f"T = {T_str:>4}  ⇒  P_A = {PA:.4f},  P_B = {PB:.4f}")     

Laufzeit: 0.159s (zwei Würfel)
Basisstrategie S = (3, 2, 1)
Insgesamt zu vergleichende Strategien: 28
Gefundene bessere Strategien: 1

T = (4, 2, 0)  ⇒  P_A = 0.4378,  P_B = 0.4466
